# LegalQA Main 2/3 — Chọn checkpoint METEOR và retrieval private

Add Input: ver3-smoke-output và output Stage 1 hoàn tất. Tập dev100 dùng trực tiếp từ snapshot Stage 1, không chia lại dataset.

MODE='auto' chọn hai checkpoint rồi retrieval private trong thời gian còn lại. MODE='select' chỉ chọn checkpoint và kết thúc paused; phiên sau gắn output đó làm PREVIOUS_OUTPUT, đặt MODE='retrieve' hoặc 'auto'. Khi gần hết ngân sách, tự lưu tiến độ. Chỉ chuyển Stage 3 khi status=complete (retrieval private đầy đủ).

Giới hạn: làm việc tối đa 9 giờ tính từ cell đầu, export tối đa 10 phút; supervisor dừng cả nhóm subprocess. Trạng thái paused là kết thúc phiên hợp lệ để Save Output và chạy tiếp. Không cam kết hoàn thành toàn bộ stage trong một phiên. Sự cố hạ tầng Kaggle vẫn có thể làm phiên kết thúc sớm.

Input test: `private-official.json` (1.918 câu). Dùng output private mới; không resume output public cũ. Tên artifact `public` trong workflow Stage 1–4 là tên nội bộ được giữ để tương thích.


In [1]:
import json, os, signal, subprocess, sys, time
from pathlib import Path

# Count setup/install time too. Do not reset this timestamp in later cells.
SESSION_STARTED = time.monotonic()
if not Path('/kaggle').is_dir():
    raise RuntimeError('Notebook chỉ chạy trên Kaggle.')
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
CODE = WORK / 'legalqa_stage_code'

# 9h includes setup and compute; export gets up to 10 additional minutes.
# The remaining margin is reserved for Kaggle output collection and runtime variation.
WORK_HOURS = 9.0
EXPORT_SECONDS = 600
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')

# None = discover exactly one matching input; set a full ROOT if multiple versions exist.
PREVIOUS_OUTPUT = None   # Cumulative output of THIS stage from an earlier session.
UPSTREAM_OUTPUT = None   # Stage 1 for notebook 02; Stage 2 for notebook 03.
LEGACY_INPUT_ROOT = None # Notebook 01 only: old legalqa_quality_v8_full with completed QLoRA.
RETRIEVAL_INPUT = None   # Stage 1: old output folder or diagnostics ZIP; reuse retrieval only.
REPO_REVISION = None     # First run: main. Continuations: automatically pin upstream commit.

STAGE = 2
MODE = 'auto'
MAX_NEW_QUESTIONS = 200


## Khóa code đúng commit và nhận diện input

Lần đầu dùng main. Các phiên tiếp theo tự checkout full SHA trong manifest, kể cả khi main đã cập nhật. Giữ một output mỗi stage trong Input hoặc đặt ROOT cụ thể.


In [2]:
if not 0 < WORK_HOURS <= 9:
    raise ValueError('WORK_HOURS phải trong (0, 9]; giữ thời gian dự phòng trước 12h.')
WORK_END = SESSION_STARTED + WORK_HOURS * 3600

def resolve_output(value, stage, required=False):
    marker = f'stage{stage}_manifest.json'
    if value is not None:
        root = Path(value)
        if not (root / marker).is_file():
            raise FileNotFoundError(root / marker)
        return root
    matches = sorted(INPUT.rglob(marker))
    if len(matches) > 1:
        raise RuntimeError(f'Nhiều output Stage {stage}: {matches}. Chỉ định ROOT ở cell cấu hình.')
    if not matches:
        if required:
            raise FileNotFoundError(f'Add Input output Stage {stage} chứa {marker}.')
        return None
    return matches[0].parent

if RETRIEVAL_INPUT is not None:
    if STAGE != 1 or any(value is not None for value in (PREVIOUS_OUTPUT, UPSTREAM_OUTPUT, LEGACY_INPUT_ROOT)):
        raise ValueError('RETRIEVAL_INPUT requires a fresh Stage 1; keep previous/upstream/legacy unset.')
    if not Path(RETRIEVAL_INPUT).exists():
        raise FileNotFoundError(RETRIEVAL_INPUT)
    # Do not auto-discover the source dataset as a resume of the old code.
else:
    PREVIOUS_OUTPUT = resolve_output(PREVIOUS_OUTPUT, STAGE)
if STAGE > 1:
    UPSTREAM_OUTPUT = resolve_output(UPSTREAM_OUTPUT, STAGE - 1, required=PREVIOUS_OUTPUT is None)
elif UPSTREAM_OUTPUT is not None:
    raise ValueError('Stage 1 không nhận UPSTREAM_OUTPUT.')
if LEGACY_INPUT_ROOT is not None:
    LEGACY_INPUT_ROOT = Path(LEGACY_INPUT_ROOT)
    if STAGE != 1 or PREVIOUS_OUTPUT is not None:
        raise ValueError('Legacy import chỉ dùng ở Stage 1 mới, không trộn với previous output.')

pins = []
for source, number in [(PREVIOUS_OUTPUT, STAGE), (UPSTREAM_OUTPUT, STAGE - 1)]:
    if source is not None:
        info = json.loads((source / f'stage{number}_manifest.json').read_text(encoding='utf-8'))
        if info.get('schema') != 2:
            raise ValueError('Input dùng schema cũ. Chọn đúng output mới hoặc legacy import ở Stage 1.')
        pins.append(info['code_commit'])
if len(set(pins)) > 1:
    raise ValueError('Upstream và previous output khác code commit.')
PIN = pins[0] if pins else (REPO_REVISION or 'main')
if pins and REPO_REVISION and REPO_REVISION != PIN:
    raise ValueError('Không đổi commit khi resume. Bắt đầu một experiment mới nếu cần đổi code.')

class BudgetPause(Exception):
    pass

def bounded_process(command, *, seconds=None, cwd=None, env=None):
    remaining = WORK_END - time.monotonic()
    if remaining <= 0:
        raise BudgetPause('Đã hết ngân sách phiên.')
    limit = remaining if seconds is None else min(remaining, seconds)
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, env=env, start_new_session=True)
    try:
        rc = process.wait(timeout=limit)
    except (subprocess.TimeoutExpired, KeyboardInterrupt) as error:
        # Worker and every legalqa subprocess share this process group.
        # Stop all of them before hashing/exporting artifacts.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            pass
        # The group leader may exit while a GPU child ignores SIGTERM.
        # Always kill remaining group members before exporting.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=30)
        if isinstance(error, KeyboardInterrupt):
            raise
        raise BudgetPause('Đã dừng worker theo ngân sách; tiến độ đã ghi sẽ được export.') from error
    if rc:
        raise subprocess.CalledProcessError(rc, command)

if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} không phải repo. Dùng phiên Kaggle mới.')
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True, timeout=30).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError('Repo origin không khớp.')
    dirty = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True, timeout=30).strip()
    if dirty:
        raise RuntimeError('Code trong session có sửa đổi; không tự ghi đè. Dùng phiên mới.')
else:
    bounded_process(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, CODE], seconds=300)
bounded_process(['git', '-C', CODE, 'fetch', '--depth', '1', 'origin', PIN], seconds=300)
bounded_process(['git', '-C', CODE, 'checkout', '--detach', 'FETCH_HEAD'], seconds=60)
commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
if pins and commit != PIN:
    raise RuntimeError('Checkout không đúng commit đã khóa.')
if not (CODE / 'legalqa' / 'stages.py').is_file():
    raise RuntimeError('Commit chưa có stages.py. Push các thay đổi mới trước khi chạy.')
print('Pinned commit:', commit)
print('Previous:', PREVIOUS_OUTPUT, '| Upstream:', UPSTREAM_OUTPUT)


Running: git clone --no-checkout --depth 1 https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git /kaggle/working/legalqa_stage_code


Cloning into '/kaggle/working/legalqa_stage_code'...


Running: git -C /kaggle/working/legalqa_stage_code fetch --depth 1 origin 6e1eb1958ba1c7ee3235860748b9eb1d177f3a57
Running: git -C /kaggle/working/legalqa_stage_code checkout --detach FETCH_HEAD


From https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa
 * branch            6e1eb1958ba1c7ee3235860748b9eb1d177f3a57 -> FETCH_HEAD


Pinned commit: 6e1eb1958ba1c7ee3235860748b9eb1d177f3a57
Previous: None | Upstream: /kaggle/input/notebooks/lighth/legalqa-main-01-qlora-train/legalqa_main_stage1_v8


HEAD is now at 6e1eb19 feat: reuse verified 5600 retrieval cache and skip BM25


## Cài môi trường và kiểm tra


In [3]:
bounded_process([sys.executable, '-m', 'pip', 'install', '-q', '-r', CODE / 'requirements.txt'], seconds=1200)
if STAGE == 2:
    bounded_process([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], seconds=300)
    bounded_process([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, seconds=300)
bounded_process([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, seconds=300)


Running: /usr/bin/python3 -m pip install -q -r /kaggle/working/legalqa_stage_code/requirements.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.
<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour


Running: /usr/bin/python3 scripts/check_metrics.py
1.0
0.9921875
Official metric runtime self-test passed; identical METEOR need not be exactly 1.0.
NLTK: 3.9.1
Running: /usr/bin/python3 -B -m unittest discover -s tests -v


test_controller_restart_polls_existing_session_without_repush (test_auto_stage1.AutoStage1Tests.test_controller_restart_polls_existing_session_without_repush) ... ok
test_failed_kaggle_run_does_not_launch_next_version (test_auto_stage1.AutoStage1Tests.test_failed_kaggle_run_does_not_launch_next_version) ... ok
test_paused_output_is_attached_automatically_and_complete_stops (test_auto_stage1.AutoStage1Tests.test_paused_output_is_attached_automatically_and_complete_stops) ... ok
test_cached_scores_preserve_fts5_ranking_ties_and_sparse_ids (test_bm25_cache.BM25CacheTests.test_cached_scores_preserve_fts5_ranking_ties_and_sparse_ids) ... ok
test_disabled_cache_uses_original_query (test_bm25_cache.BM25CacheTests.test_disabled_cache_uses_original_query) ... ok
test_warm_query_needs_no_sql_and_cache_stays_bounded (test_bm25_cache.BM25CacheTests.test_warm_query_needs_no_sql_and_cache_stays_bounded) ... ok
test_answer_flags_ignore_dates_and_substantive_information_clauses (test_core.CoreTests.te

Starting session 1: example/legalqa-test-f740c0e9-001; previous=None
example/legalqa-test-f740c0e9-001: RUNNING
example/legalqa-test-f740c0e9-001: COMPLETE
Stage 1 session 1: complete
Stage 1 complete. Attach example/legalqa-test-f740c0e9-001 output to Stage 2.
Starting session 1: example/legalqa-test-9e91fca9-001; previous=None
example/legalqa-test-9e91fca9-001: ERROR
Starting session 1: example/legalqa-test-1a74f491-001; previous=None
example/legalqa-test-1a74f491-001: COMPLETE
Stage 1 session 1: paused
Starting session 2: example/legalqa-test-1a74f491-002; previous=example/legalqa-test-1a74f491-001
example/legalqa-test-1a74f491-002: COMPLETE
Stage 1 session 2: complete
Stage 1 complete. Attach example/legalqa-test-1a74f491-002 output to Stage 2.


...

Progress: {'complete': False, 'phase': 'public_retrieval'}
SNAPSHOT paused: /tmp/tmpd6ui8xa4/second/run/stage2_manifest.json
Diagnostics: /tmp/tmpd6ui8xa4/second/run/legalqa_main_stage2_v8_diagnostics.zip
Evaluation objective: {'primary_metric': 'meteor', 'secondary_metric': 'rougeL', 'target_meteor': 0.65}
Retrieval: {'bm25_cache_mb': 1024, 'fast_phrase_precise': True, 'phrase_cache_mb': 64, 'phrase_workers': 2, 'phrase_reuse_readers': True, 'phrase_mmap_mb': 1024, 'bm25_k': 100, 'dense_k': 100, 'rrf_constant': 60, 'precise_bm25_k': 40, 'pool_k': 32, 'max_children_per_parent': 2, 'parents_k': 4, 'lexical_score_weight': 2.0, 'phrase_match_bonus': 1.0, 'exact_document_bonus': 4.0, 'year_match_bonus': 1.0, 'recency_bonus': 0.75, 'embedding_batch': 32, 'reranker_batch': 8, 'reranker_max_tokens': 768} Generation: {'max_input_tokens': 4096, 'max_new_tokens': 1536, 'parent_max_tokens': 1400, 'min_context_tokens': 256, 'load_in_4bit': True}
Progress: {'complete': True, 'phase': 'public_ready'

ok
test_stage_launches_torchrun_only_for_fit (test_stages.StageTests.test_stage_launches_torchrun_only_for_fit) ... ok
test_three_notebooks_share_supervisor_setup_and_runner (test_stages.StageTests.test_three_notebooks_share_supervisor_setup_and_runner) ... ok
test_timeout_cannot_publish_complete_from_old_progress (test_stages.StageTests.test_timeout_cannot_publish_complete_from_old_progress) ... ok
test_training_callback_saves_once_and_keeps_completed_epoch (test_stages.StageTests.test_training_callback_saves_once_and_keeps_completed_epoch) ... ok
test_training_reuse_rejects_changed_config_and_split (test_stages.StageTests.test_training_reuse_rejects_changed_config_and_split) ... ok
test_training_uses_two_workers_with_unchanged_effective_batch (test_stages.StageTests.test_training_uses_two_workers_with_unchanged_effective_batch) ... ok
test_fp16_frozen_head_loss_gradients_and_global_token_denominator (test_training_memory.FusedLossCudaTests.test_fp16_frozen_head_loss_gradients_and_glo

QLoRA loss=liger_fused_linear_cross_entropy; full targets; no CPU offload
QLoRA loss=liger_fused_linear_cross_entropy; full targets; no CPU offload


ok
test_compaction_preserves_targets_masks_order_and_padding (test_training_memory.TrainingMemoryTests.test_compaction_preserves_targets_masks_order_and_padding) ... ok
test_patch_is_instance_local_and_preserves_loss_kwargs (test_training_memory.TrainingMemoryTests.test_patch_is_instance_local_and_preserves_loss_kwargs) ... ok
test_rejects_unsupported_head_before_training (test_training_memory.TrainingMemoryTests.test_rejects_unsupported_head_before_training) ... ok

----------------------------------------------------------------------
Ran 104 tests in 32.758s

OK


QLoRA loss=liger_fused_linear_cross_entropy; full targets; no CPU offload


## Chạy trong ngân sách và export cả tiến độ dở dang

Mọi xử lý/kiểm tra artifact dùng legalqa/stages.py chung cho ba notebook. Diagnostics chứa dữ liệu dev, reference, retrieval, prediction, audit, metrics và trạng thái train đã có; không chứa trọng số.


In [4]:
if 'private-official.json' not in (CODE / 'legalqa/stages.py').read_text(encoding='utf-8'):
    raise RuntimeError('Pinned code still uses public data. Push the private update and start a new run; do not resume old public outputs.')

RUN_ROOT = WORK / f'legalqa_main_stage{STAGE}_v8'
OPTIONS = WORK / f'legalqa_stage{STAGE}_options.json'
options = {
    'stage': STAGE, 'root': str(RUN_ROOT), 'version3': str(VERSION3_ROOT),
    'dataset': str(DATASET_ROOT), 'mode': MODE, 'max_new_questions': MAX_NEW_QUESTIONS,
    'previous': str(PREVIOUS_OUTPUT) if PREVIOUS_OUTPUT is not None else None,
    'upstream': str(UPSTREAM_OUTPUT) if UPSTREAM_OUTPUT is not None else None,
    'legacy': str(LEGACY_INPUT_ROOT) if LEGACY_INPUT_ROOT is not None else None,
    'retrieval_input': str(RETRIEVAL_INPUT) if RETRIEVAL_INPUT is not None else None,
}
OPTIONS.write_text(json.dumps(options, ensure_ascii=False, indent=2), encoding='utf-8')
# Cooperative pause 10 minutes before hard worker stop, for saving Trainer state.
remaining = max(0, WORK_END - time.monotonic())
worker_env = {**os.environ, 'LEGALQA_DEADLINE': str(time.time() + max(0, remaining - 600)),
              'PYTHONUNBUFFERED': '1'}
outcome, failure = 'ok', None
try:
    bounded_process([sys.executable, '-m', 'legalqa.stages', 'run', '--options', OPTIONS],
                    cwd=CODE, env=worker_env)
except BudgetPause as error:
    outcome = 'paused'
    print(str(error), flush=True)
except Exception as error:
    outcome, failure = 'failed', error
finally:
    if (RUN_ROOT / 'session.json').is_file():
        # Export runs only after the entire compute process group has stopped.
        # It is bounded separately, without restarting the 9h compute budget.
        original_end = WORK_END
        WORK_END = min(SESSION_STARTED + 10 * 3600, time.monotonic() + EXPORT_SECONDS)
        try:
            bounded_process([sys.executable, '-m', 'legalqa.stages', 'finalize',
                             '--options', OPTIONS, '--outcome', outcome], cwd=CODE)
        finally:
            WORK_END = original_end
if failure is not None:
    raise failure
manifest_path = RUN_ROOT / f'stage{STAGE}_manifest.json'
if not manifest_path.is_file():
    raise RuntimeError('Chưa tạo được snapshot; xem lỗi setup ở trên.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('STATUS:', manifest['status'])
print('PROGRESS:', manifest['progress'])
print('OUTPUT:', RUN_ROOT)
if manifest['status'] == 'complete':
    print('Stage hoàn tất. Có thể dùng output cho stage tiếp theo.')
else:
    print('Phiên kết thúc có chủ đích. Save output, Add Input vào CÙNG notebook, rồi chạy tiếp.')
print('submission.zip chỉ được tạo khi đủ tất cả ID private, không nộp partial JSON.')


Running: /usr/bin/python3 -m legalqa.stages run --options /kaggle/working/legalqa_stage2_options.json
Evaluation objective: {'primary_metric': 'meteor', 'secondary_metric': 'rougeL', 'target_meteor': 0.65}
Retrieval: {'bm25_cache_mb': 1024, 'fast_phrase_precise': True, 'phrase_cache_mb': 64, 'phrase_workers': 2, 'phrase_reuse_readers': True, 'phrase_mmap_mb': 1024, 'bm25_k': 100, 'dense_k': 100, 'rrf_constant': 60, 'precise_bm25_k': 40, 'pool_k': 32, 'max_children_per_parent': 2, 'parents_k': 4, 'lexical_score_weight': 2.0, 'phrase_match_bonus': 1.0, 'exact_document_bonus': 4.0, 'year_match_bonus': 1.0, 'recency_bonus': 0.75, 'embedding_batch': 32, 'reranker_batch': 8, 'reranker_max_tokens': 768} Generation: {'max_input_tokens': 4096, 'max_new_tokens': 1536, 'parent_max_tokens': 1400, 'min_context_tokens': 256, 'load_in_4bit': True}
Running: /usr/bin/python3 -m legalqa --config /kaggle/working/legalqa_main_stage2_v8/config.json --models /kaggle/working/stage2_runtime_models retrieve --

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Retrieval mode=full, bm25_k=100, precise_bm25_k=40, bm25_cache_mb=1024, fast_phrase_precise=True, phrase_workers=2, phrase_cache_mb=64
Phrase SQLite: reusable_readers=2, mmap_bytes=[1073741824, 1073741824]
Retrieved: 10/100
Retrieved: 20/100
Retrieved: 30/100
Retrieved: 40/100
Retrieved: 50/100
Retrieved: 60/100
Retrieved: 70/100
Retrieved: 80/100
Retrieved: 90/100
Retrieved: 100/100
{
  "records": 100,
  "output": "/kaggle/working/legalqa_main_stage2_v8/dev100.retrieval.json",
  "fingerprint": "ec5b7e7552d75a3f3bf89870114ca8887e55068718c9f4e99571fa11c7666e20"
}
Running: /usr/bin/python3 -m legalqa --config /kaggle/working/legalqa_main_stage2_v8/config.json --models /kaggle/working/stage2_runtime_models generate --questions /kaggle/working/legalqa_main_stage2_v8/data/dev100.questions.json --retrieval /kaggle/working/legalqa_main_stage2_v8/dev100.retrieval.json --output /kaggle/working/legalqa_main_stage2_v8/dev100.base_v8.json


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [01:10<00:00, 35.03s/it]
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based genera

Answered: 10/100
Answered: 20/100
Answered: 30/100
Answered: 40/100
Answered: 50/100
Answered: 60/100
Answered: 70/100
Answered: 80/100
Answered: 90/100
Answered: 100/100
{
  "samples": 100,
  "output": "/kaggle/working/legalqa_main_stage2_v8/dev100.base_v8.json"
}
Running: /usr/bin/python3 -m legalqa --config /kaggle/working/legalqa_main_stage2_v8/config.json --models /kaggle/working/stage2_runtime_models evaluate --predictions /kaggle/working/legalqa_main_stage2_v8/dev100.base_v8.json --references /kaggle/working/legalqa_main_stage2_v8/data/dev100.references.json --output /kaggle/working/legalqa_main_stage2_v8/dev100.base.metrics.json --label base_v8
{
  "label": "base_v8",
  "samples": 100,
  "meteor": 0.46177589545277103,
  "rougeL": 0.5084392523232798,
  "objective": {
    "primary_metric": "meteor",
    "secondary_metric": "rougeL",
    "target_meteor": 0.65,
    "target_met": false,
    "meteor_gap": 0.188224104547229
  },
  "prediction_path": "/kaggle/working/legalqa_main_stage

...

Answered: 10/100
Answered: 20/100
Answered: 30/100
Answered: 40/100
Answered: 50/100
Answered: 60/100
Answered: 70/100
Answered: 80/100
Answered: 90/100
Answered: 100/100
{
  "samples": 100,
  "output": "/kaggle/working/legalqa_main_stage2_v8/dev100.epoch-01.json"
}
Running: /usr/bin/python3 -m legalqa --config /kaggle/working/legalqa_main_stage2_v8/config.json --models /kaggle/working/stage2_runtime_models evaluate --predictions /kaggle/working/legalqa_main_stage2_v8/dev100.epoch-01.json --references /kaggle/working/legalqa_main_stage2_v8/data/dev100.references.json --output /kaggle/working/legalqa_main_stage2_v8/dev100.epoch-01.metrics.json --label epoch-01
{
  "label": "epoch-01",
  "samples": 100,
  "meteor": 0.6182826973445281,
  "rougeL": 0.5908622414672994,
  "objective": {
    "primary_metric": "meteor",
    "secondary_metric": "rougeL",
    "target_meteor": 0.65,
    "target_met": false,
    "meteor_gap": 0.0317173026554719
  },
  "prediction_path": "/kaggle/working/legalqa_ma

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.13s/it]
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based genera

Answered: 10/100
Answered: 20/100
Answered: 30/100
Answered: 40/100
Answered: 50/100
Answered: 60/100
Answered: 70/100
Answered: 80/100
Answered: 90/100
Answered: 100/100
{
  "samples": 100,
  "output": "/kaggle/working/legalqa_main_stage2_v8/dev100.epoch-02.json"
}
Running: /usr/bin/python3 -m legalqa --config /kaggle/working/legalqa_main_stage2_v8/config.json --models /kaggle/working/stage2_runtime_models evaluate --predictions /kaggle/working/legalqa_main_stage2_v8/dev100.epoch-02.json --references /kaggle/working/legalqa_main_stage2_v8/data/dev100.references.json --output /kaggle/working/legalqa_main_stage2_v8/dev100.epoch-02.metrics.json --label epoch-02
{
  "label": "epoch-02",
  "samples": 100,
  "meteor": 0.602640747541288,
  "rougeL": 0.5893317438640158,
  "objective": {
    "primary_metric": "meteor",
    "secondary_metric": "rougeL",
    "target_meteor": 0.65,
    "target_met": false,
    "meteor_gap": 0.047359252458711976
  },
  "prediction_path": "/kaggle/working/legalqa_m

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Retrieval mode=full, bm25_k=100, precise_bm25_k=40, bm25_cache_mb=1024, fast_phrase_precise=True, phrase_workers=2, phrase_cache_mb=64
Phrase SQLite: reusable_readers=2, mmap_bytes=[1073741824, 1073741824]
Retrieved: 10/1000
Retrieved: 20/1000
Retrieved: 30/1000
Retrieved: 40/1000
Retrieved: 50/1000
Retrieved: 60/1000
Retrieved: 70/1000
Retrieved: 80/1000
Retrieved: 90/1000
Retrieved: 100/1000
Retrieved: 110/1000
Retrieved: 120/1000
Retrieved: 130/1000
Retrieved: 140/1000
Retrieved: 150/1000
Retrieved: 160/1000
Retrieved: 170/1000
Retrieved: 180/1000
Retrieved: 190/1000
Retrieved: 200/1000
Retrieved: 210/1000
Retrieved: 220/1000
Retrieved: 230/1000
Retrieved: 240/1000
Retrieved: 250/1000
Retrieved: 260/1000
Retrieved: 270/1000
Retrieved: 280/1000
Retrieved: 290/1000
Retrieved: 300/1000
Retrieved: 310/1000
Retrieved: 320/1000
Retrieved: 330/1000
Retrieved: 340/1000
Retrieved: 350/1000
Retrieved: 360/1000
Retrieved: 370/1000
Retrieved: 380/1000
Retrieved: 390/1000
Retrieved: 400/1000
Ret